<a href="https://colab.research.google.com/github/nehalnady/DM_Project/blob/main/DM_Classification_Task_Updated_Finished.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Task: Customer Churn Prediction
## Dataset: Telco Customer Churn (Kaggle)

**Goal:** Predict whether a customer will leave the telecom company (Churn = Yes/No).

### Steps
1. Load & Explore Data
2. Preprocessing (Duplicates, Missing, Encoding, Scaling)
3. Handle Class Imbalance (SMOTE)
4. Feature Selection
5. Train/Test Split (80/20)
6. Train Random Forest Classifier
7. Cross-Validation (5-Fold)
8. Evaluate (Accuracy, Precision, Recall, F1, ROC-AUC)
9. Visualizations
10. Save Model
11. Interactive Prediction (Step 23)


In [2]:
# STEP 1: Install & Import Libraries
# ---------------------------------------------------------------
# Uncomment the next line if running in Google Colab:
# !pip install -q scikit-learn imbalanced-learn pandas numpy matplotlib seaborn joblib

import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend (works in Colab & Jupyter)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection    import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing      import LabelEncoder, StandardScaler
from sklearn.impute             import SimpleImputer
from sklearn.ensemble           import RandomForestClassifier
from sklearn.feature_selection  import VarianceThreshold
from sklearn.metrics            import (accuracy_score, precision_score, recall_score,
                                         f1_score, roc_auc_score, confusion_matrix,
                                         classification_report, roc_curve)
from imblearn.over_sampling     import SMOTE
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('All libraries imported successfully.')


All libraries imported successfully.


In [3]:
# STEP 2: Load Dataset
# ---------------------------------------------------------------
# Place "WA_Fn-UseC_-Telco-Customer-Churn.csv" in the same folder as this notebook.
# In Colab: upload the file or connect your Google Drive.

# DATASET_NAME = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
# SEARCH_DIRS  = ['.', '/content/', '/content/drive/MyDrive/',
#                 '/content/drive/MyDrive/DM_Project/']

# df = None
# for d in SEARCH_DIRS:
#     path = os.path.join(d, DATASET_NAME)
#     if os.path.exists(path):
#         df = pd.read_csv(path)
#         print(f'Loaded from: {path}')
#         break

# if df is None:
#     raise FileNotFoundError(
#         f'{DATASET_NAME} not found.\n'
#         'Upload it to Colab or place it in the same folder as this notebook.')



df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()


Shape: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# STEP 3: Initial Exploration
# ---------------------------------------------------------------
print('='*55)
print('STEP 3: Basic Data Exploration')
print('='*55)
print(f'  Rows      : {df.shape[0]}')
print(f'  Columns   : {df.shape[1]}')
print(f'  Duplicates: {df.duplicated().sum()}')
print('\nMissing Values:')
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() or '  None')
print('\nChurn Distribution:')
print(df['Churn'].value_counts().to_string())
print(df.dtypes.to_string())


STEP 3: Basic Data Exploration
  Rows      : 7043
  Columns   : 21
  Duplicates: 0

Missing Values:
Series([], )

Churn Distribution:
Churn
No     5174
Yes    1869
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object


In [5]:
# STEP 4: Preprocessing
# ---------------------------------------------------------------
# 4a. Drop customerID (not a feature)
df.drop(columns=['customerID'], errors='ignore', inplace=True)

# 4b. Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# 4c. Fix TotalCharges (spaces -> numeric)
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 4d. Separate features and target
y = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
X = df.drop(columns=['Churn'])

# 4e. Identify column types
binary_cols = [c for c in X.columns
               if X[c].nunique() == 2 and X[c].dtype == 'object']
multi_cols  = [c for c in X.select_dtypes(include='object').columns
               if c not in binary_cols]
num_cols    = X.select_dtypes(include=['int64','float64']).columns.tolist()

print(f'Binary columns   : {binary_cols}')
print(f'Multiclass cols  : {multi_cols}')
print(f'Numeric cols     : {num_cols}')


Duplicates removed: 22
Binary columns   : ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Multiclass cols  : ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
Numeric cols     : ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [6]:
# STEP 5: Impute Missing Values
# ---------------------------------------------------------------
if num_cols:
    X[num_cols] = SimpleImputer(strategy='median').fit_transform(X[num_cols])
if binary_cols or multi_cols:
    cat_all = binary_cols + multi_cols
    X[cat_all] = SimpleImputer(strategy='most_frequent').fit_transform(X[cat_all])
print('Missing values imputed.')


Missing values imputed.


In [7]:
# STEP 6: Outlier Handling (IQR clipping)
# ---------------------------------------------------------------
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    if col in X.columns:
        Q1, Q3 = X[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        X[col] = X[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
print('Outlier clipping done.')


Outlier clipping done.


In [8]:
# STEP 7: Encoding
# ---------------------------------------------------------------
# Label Encoding for binary columns
label_encoders = {}
for col in binary_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# One-Hot Encoding for multi-class columns
X = pd.get_dummies(X, columns=multi_cols, drop_first=True)

print(f'X shape after encoding: {X.shape}')


X shape after encoding: (7021, 30)


In [9]:
# STEP 8: Feature Names & Scaling
# ---------------------------------------------------------------
feature_names = list(X.columns)
num_scale_cols = [c for c in num_cols if c in X.columns]

scaler = StandardScaler()
X[num_scale_cols] = scaler.fit_transform(X[num_scale_cols])
print(f'Scaled columns: {num_scale_cols}')
print(f'Total features: {len(feature_names)}')


Scaled columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Total features: 30


In [10]:
# STEP 9: Correlation Heatmap (top 12 numeric features)
# ---------------------------------------------------------------
numeric_df = pd.concat([X.select_dtypes(include=['float64','int64']), y.rename('Churn')], axis=1)
if numeric_df.shape[1] > 2:
    top_feats = numeric_df.corr()['Churn'].abs().sort_values(ascending=False).head(12).index
    plt.figure(figsize=(10, 7))
    sns.heatmap(numeric_df[top_feats].corr(), annot=True, fmt='.2f',
                cmap='coolwarm', center=0, linewidths=0.5)
    plt.title('Correlation Heatmap (Top Features vs Churn)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('classification_heatmap.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved: classification_heatmap.png')


Saved: classification_heatmap.png


In [11]:
# STEP 10: Boxplots of Numeric Features by Churn
# ---------------------------------------------------------------
plot_num = [c for c in ['tenure','MonthlyCharges','TotalCharges'] if c in X.columns]
if plot_num:
    fig, axes = plt.subplots(1, len(plot_num), figsize=(5*len(plot_num), 4))
    if len(plot_num) == 1:
        axes = [axes]
    for ax, col in zip(axes, plot_num):
        tmp = pd.DataFrame({col: X[col], 'Churn': y})
        sns.boxplot(data=tmp, x='Churn', y=col, palette='Set2', ax=ax)
        ax.set_title(col, fontsize=11, fontweight='bold')
        ax.set_xticklabels(['No Churn', 'Churn'])
    plt.suptitle('Feature Distributions by Churn Status', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('classification_boxplots.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved: classification_boxplots.png')


Saved: classification_boxplots.png


In [12]:
# STEP 11: Train/Test Split
# ---------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train Churn rate: {y_train.mean():.3f}')
print(f'Test  Churn rate: {y_test.mean():.3f}')


Train: (5616, 30)  |  Test: (1405, 30)
Train Churn rate: 0.264
Test  Churn rate: 0.265


In [13]:
# STEP 12: SMOTE - Balance Training Data
# ---------------------------------------------------------------
# SMOTE synthesizes new minority-class samples so the model
# doesn't learn to always predict the majority class.
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'Before SMOTE: {y_train.value_counts().to_dict()}')
print(f'After  SMOTE: {dict(zip(*np.unique(y_train_res, return_counts=True)))}')


Before SMOTE: {0: 4131, 1: 1485}
After  SMOTE: {np.int64(0): np.int64(4131), np.int64(1): np.int64(4131)}


In [14]:
# STEP 13: Feature Selection (Variance Threshold)
# ---------------------------------------------------------------
selector = VarianceThreshold(threshold=0.01)
X_train_sel = selector.fit_transform(X_train_res)
X_test_sel  = selector.transform(X_test)
selected_features = [feature_names[i] for i in selector.get_support(indices=True)]
X_train_sel = pd.DataFrame(X_train_sel, columns=selected_features)
X_test_sel  = pd.DataFrame(X_test_sel,  columns=selected_features)
print(f'Features before: {X_train_res.shape[1]}')
print(f'Features after : {X_train_sel.shape[1]}')


Features before: 30
Features after : 30


In [15]:
# STEP 14: Train Random Forest Classifier
# ---------------------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_sel, y_train_res)
print('Random Forest trained!')
print(f'  Trees : {rf.n_estimators}')
print(f'  Depth : {rf.max_depth}')


Random Forest trained!
  Trees : 200
  Depth : 10


In [16]:
# STEP 15: Cross-Validation (5-Fold Stratified)
# ---------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_f1 = cross_val_score(rf, X_train_sel, y_train_res, cv=cv, scoring='f1', n_jobs=-1)
cv_acc = cross_val_score(rf, X_train_sel, y_train_res, cv=cv, scoring='accuracy', n_jobs=-1)
print('='*50)
print('STEP 15: Cross-Validation (5-Fold)')
print('='*50)
print('  F1 Scores    :', cv_f1.round(4).tolist())
print('  F1 Mean      :', round(cv_f1.mean(), 4), '+/-', round(cv_f1.std(), 4))
print('  Acc Mean     :', round(cv_acc.mean(), 4))


STEP 15: Cross-Validation (5-Fold)
  F1 Scores    : [0.817, 0.8272, 0.8197, 0.8285, 0.8342]
  F1 Mean      : 0.8253 +/- 0.0062
  Acc Mean     : 0.8188


In [17]:
# STEP 16: Evaluate on Test Set
# ---------------------------------------------------------------
y_pred = rf.predict(X_test_sel)
y_prob = rf.predict_proba(X_test_sel)[:, 1]

acc   = accuracy_score(y_test, y_pred)
prec  = precision_score(y_test, y_pred)
rec   = recall_score(y_test, y_pred)
f1    = f1_score(y_test, y_pred)
auc   = roc_auc_score(y_test, y_prob)

print('='*50)
print('STEP 16: Test Set Evaluation')
print('='*50)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  ROC-AUC   : {auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Churn','Churn']))


STEP 16: Test Set Evaluation
  Accuracy  : 0.7658
  Precision : 0.5440
  Recall    : 0.7151
  F1-Score  : 0.6179
  ROC-AUC   : 0.8359

              precision    recall  f1-score   support

    No Churn       0.88      0.78      0.83      1033
       Churn       0.54      0.72      0.62       372

    accuracy                           0.77      1405
   macro avg       0.71      0.75      0.72      1405
weighted avg       0.79      0.77      0.77      1405



In [18]:
# STEP 17: Confusion Matrix
# ---------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn','Churn'],
            yticklabels=['No Churn','Churn'])
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('classification_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: classification_confusion_matrix.png')


Saved: classification_confusion_matrix.png


In [19]:
# STEP 18: ROC Curve
# ---------------------------------------------------------------
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='royalblue', linewidth=2.5, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0,1],[0,1], color='grey', linestyle='--')
plt.xlabel('False Positive Rate', fontsize=11)
plt.ylabel('True Positive Rate', fontsize=11)
plt.title('ROC Curve', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('classification_roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: classification_roc_curve.png')


Saved: classification_roc_curve.png


In [20]:
# STEP 19: Feature Importance
# ---------------------------------------------------------------
importances = pd.Series(rf.feature_importances_, index=selected_features)
top15 = importances.sort_values(ascending=False).head(15)
plt.figure(figsize=(9, 6))
colors = plt.cm.RdYlGn(np.linspace(0.9, 0.35, len(top15)))
plt.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1])
plt.xlabel('Importance Score', fontsize=11)
plt.title('Top 15 Feature Importances', fontsize=13, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('classification_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: classification_feature_importance.png')


Saved: classification_feature_importance.png


In [21]:
# STEP 20: Churn Distribution Plot
# ---------------------------------------------------------------
churn_rate = y.mean()
plt.figure(figsize=(5, 4))
bars = plt.bar(['No Churn','Churn'], [1-churn_rate, churn_rate],
               color=['steelblue','tomato'], edgecolor='white')
for bar, val in zip(bars, [1-churn_rate, churn_rate]):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{val*100:.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.title('Target Class Distribution', fontsize=13, fontweight='bold')
plt.ylabel('Proportion')
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('classification_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: classification_distribution.png')


Saved: classification_distribution.png


In [22]:
# STEP 21: Summary
# ---------------------------------------------------------------
print('='*55)
print('CLASSIFICATION RESULTS SUMMARY')
print('='*55)
print(f'  Dataset rows   : {len(df)}')
print(f'  Features used  : {len(selected_features)}')
print(f'  Model          : RandomForestClassifier (200 trees)')
print(f'  ---- Test Performance ----')
print(f'  Accuracy       : {acc:.4f}')
print(f'  F1-Score       : {f1:.4f}')
print(f'  ROC-AUC        : {auc:.4f}')
print(f'  CV F1 Mean     : {cv_f1.mean():.4f}')
print('='*55)


CLASSIFICATION RESULTS SUMMARY
  Dataset rows   : 7021
  Features used  : 30
  Model          : RandomForestClassifier (200 trees)
  ---- Test Performance ----
  Accuracy       : 0.7658
  F1-Score       : 0.6179
  ROC-AUC        : 0.8359
  CV F1 Mean     : 0.8253


In [23]:
# STEP 22: Save Model
# ---------------------------------------------------------------
churn_model_data = {
    'model'            : rf,
    'scaler'           : scaler,
    'feature_selector' : selector,
    'feature_names'    : feature_names,
    'selected_features': selected_features,
    'num_scale_cols'   : num_scale_cols,
    'multi_cols'       : multi_cols,
    'label_encoders'   : label_encoders,
}
joblib.dump(churn_model_data, 'churn_model.pkl')
print('churn_model.pkl saved!')


churn_model.pkl saved!


---
## Step 23: Interactive Churn Prediction

Use the widgets below to test the model with any values.
Click **Load Sample A**, **B**, or **C** to instantly pre-fill all fields, then click **Predict Churn**.

| Sample | Description | Expected Result |
|--------|-------------|-----------------|
| A | High-Risk: Month-to-month contract, Fiber optic, no security | ~Churn |
| B | Low-Risk: 2-year contract, DSL, all security + support services | ~No Churn |
| C | Medium-Risk: 1-year contract, Fiber optic, some services | Borderline |


In [24]:
# STEP 23: Interactive Churn Predictor (Works in Colab & Jupyter)
# ---------------------------------------------------------------
try:
    import ipywidgets as widgets
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])
    import ipywidgets as widgets

from IPython.display import display, clear_output
import pandas as pd, numpy as np, joblib

# Load model (must have run Step 22 above)
try:
    _churn = joblib.load('churn_model.pkl')
    print('Model loaded. Use the form below to test predictions.')
except FileNotFoundError:
    raise FileNotFoundError('Run all cells above first to generate churn_model.pkl')

# ---- Built-in sample test cases ----
SAMPLES = {
    'A - High Risk (likely Churn)': dict(
        gender='Male', SeniorCitizen='Yes', Partner='No', Dependents='No',
        tenure='3', MonthlyCharges='95.50', TotalCharges='286.50',
        PhoneService='Yes', MultipleLines='No', InternetService='Fiber optic',
        OnlineSecurity='No', OnlineBackup='No', DeviceProtection='No',
        TechSupport='No', StreamingTV='No', StreamingMovies='No',
        Contract='Month-to-month', PaperlessBilling='Yes',
        PaymentMethod='Electronic check'),
    'B - Low Risk (likely No Churn)': dict(
        gender='Female', SeniorCitizen='No', Partner='Yes', Dependents='Yes',
        tenure='52', MonthlyCharges='54.00', TotalCharges='2808.00',
        PhoneService='Yes', MultipleLines='Yes', InternetService='DSL',
        OnlineSecurity='Yes', OnlineBackup='Yes', DeviceProtection='Yes',
        TechSupport='Yes', StreamingTV='No', StreamingMovies='No',
        Contract='Two year', PaperlessBilling='No',
        PaymentMethod='Credit card (automatic)'),
    'C - Medium Risk': dict(
        gender='Male', SeniorCitizen='No', Partner='No', Dependents='No',
        tenure='18', MonthlyCharges='78.75', TotalCharges='1417.50',
        PhoneService='Yes', MultipleLines='No', InternetService='Fiber optic',
        OnlineSecurity='No', OnlineBackup='Yes', DeviceProtection='No',
        TechSupport='No', StreamingTV='Yes', StreamingMovies='Yes',
        Contract='One year', PaperlessBilling='Yes',
        PaymentMethod='Mailed check'),
}

# ---- Create widgets ----
def dd(opts, val=None):
    return widgets.Dropdown(options=opts, value=val or opts[0],
                            layout=widgets.Layout(width='210px'))
def txt(val=''):
    return widgets.Text(value=val, layout=widgets.Layout(width='210px'))
def r(label, w):
    return widgets.HBox([widgets.Label(label, layout=widgets.Layout(width='175px')), w])

W = dict(
    gender          = dd(['Male','Female']),
    SeniorCitizen   = dd(['No','Yes']),
    Partner         = dd(['Yes','No']),
    Dependents      = dd(['Yes','No']),
    tenure          = txt('12'),
    MonthlyCharges  = txt('65.00'),
    TotalCharges    = txt('780.00'),
    PhoneService    = dd(['Yes','No']),
    MultipleLines   = dd(['No phone service','No','Yes']),
    InternetService = dd(['DSL','Fiber optic','No']),
    OnlineSecurity  = dd(['No','Yes','No internet service']),
    OnlineBackup    = dd(['No','Yes','No internet service']),
    DeviceProtection= dd(['No','Yes','No internet service']),
    TechSupport     = dd(['No','Yes','No internet service']),
    StreamingTV     = dd(['No','Yes','No internet service']),
    StreamingMovies = dd(['No','Yes','No internet service']),
    Contract        = dd(['Month-to-month','One year','Two year']),
    PaperlessBilling= dd(['Yes','No']),
    PaymentMethod   = dd(['Electronic check','Mailed check',
                          'Bank transfer (automatic)','Credit card (automatic)']),
)

def load_sample(key):
    for k, v in SAMPLES[key].items():
        if k in W: W[k].value = v

btn_A = widgets.Button(description='Load Sample A', button_style='danger',  layout=widgets.Layout(width='145px'))
btn_B = widgets.Button(description='Load Sample B', button_style='success', layout=widgets.Layout(width='145px'))
btn_C = widgets.Button(description='Load Sample C', button_style='warning', layout=widgets.Layout(width='145px'))
btn_A.on_click(lambda b: load_sample('A - High Risk (likely Churn)'))
btn_B.on_click(lambda b: load_sample('B - Low Risk (likely No Churn)'))
btn_C.on_click(lambda b: load_sample('C - Medium Risk'))

btn_predict = widgets.Button(description='Predict Churn', button_style='primary',
                             layout=widgets.Layout(width='150px', height='38px'))
out = widgets.Output()

def on_predict(b):
    with out:
        clear_output(wait=True)
        try:
            raw = {k: ww.value for k, ww in W.items()}
            for key in ['tenure','MonthlyCharges','TotalCharges']:
                raw[key] = float(raw[key])
            raw['SeniorCitizen'] = 1.0 if raw['SeniorCitizen'] == 'Yes' else 0.0
            for col, mp in [('gender',{'Male':1,'Female':0}),
                            ('Partner',{'Yes':1,'No':0}),
                            ('Dependents',{'Yes':1,'No':0}),
                            ('PhoneService',{'Yes':1,'No':0}),
                            ('PaperlessBilling',{'Yes':1,'No':0})]:
                raw[col] = mp.get(raw[col], 0)
            df_r = pd.DataFrame([raw])
            df_r = pd.get_dummies(df_r, columns=_churn['multi_cols'], drop_first=True)
            for c in _churn['feature_names']:
                if c not in df_r.columns: df_r[c] = 0
            df_r = df_r.reindex(columns=_churn['feature_names'], fill_value=0)
            df_r[_churn['num_scale_cols']] = _churn['scaler'].transform(df_r[_churn['num_scale_cols']])
            X_s = pd.DataFrame(_churn['feature_selector'].transform(df_r),
                               columns=_churn['selected_features'])
            prob  = _churn['model'].predict_proba(X_s)[0][1]
            label = 'CHURN' if prob >= 0.5 else 'NO CHURN'
            bar   = '#' * int(prob * 25) + '.' * (25 - int(prob * 25))
            print('='*35)
            print(' Prediction  :', label)
            print(' Churn Prob  :', round(prob * 100, 1), '%')
            print(' [' + bar + ']')
            print('='*35)
        except Exception as e:
            print('ERROR:', e)

btn_predict.on_click(on_predict)

col1 = widgets.VBox([widgets.HTML('<b>Account Info</b>'),
    r('Gender',           W['gender']),
    r('Senior Citizen',   W['SeniorCitizen']),
    r('Partner',          W['Partner']),
    r('Dependents',       W['Dependents']),
    r('Tenure (months)',  W['tenure']),
    r('Monthly Charges',  W['MonthlyCharges']),
    r('Total Charges',    W['TotalCharges']),
])
col2 = widgets.VBox([widgets.HTML('<b>Services & Billing</b>'),
    r('Phone Service',     W['PhoneService']),
    r('Multiple Lines',    W['MultipleLines']),
    r('Internet Service',  W['InternetService']),
    r('Online Security',   W['OnlineSecurity']),
    r('Online Backup',     W['OnlineBackup']),
    r('Device Protection', W['DeviceProtection']),
    r('Tech Support',      W['TechSupport']),
    r('Streaming TV',      W['StreamingTV']),
    r('Streaming Movies',  W['StreamingMovies']),
    r('Contract',          W['Contract']),
    r('Paperless Billing', W['PaperlessBilling']),
    r('Payment Method',    W['PaymentMethod']),
])
display(widgets.VBox([
    widgets.HTML('<h3>Customer Churn Predictor</h3>'),
    widgets.HTML('<b>Step 1:</b> Load a sample or fill fields manually'),
    widgets.HBox([btn_A, btn_B, btn_C]),
    widgets.HTML('<br>'),
    widgets.HBox([col1, widgets.HTML('&nbsp;&nbsp;'), col2]),
    widgets.HTML('<br>'),
    widgets.HTML('<b>Step 2:</b> Click Predict'),
    btn_predict,
    out,
]))


Model loaded. Use the form below to test predictions.
